# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

The dataset contains ordered logistic regression results related to the adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps us understand the structure of the dataset, including which record sets (tables) and fields (columns) are available for analysis.

We will explore the dataset metadata for available record sets, and for each record set, examine its fields referenced by `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = []
for record_set in metadata.record_sets:
    print(f"Record Set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    fields = getattr(record_set, 'fields', [])
    if not fields:
        print("    (No fields listed)")
    else:
        for field in fields:
            print(f"    Field: {getattr(field, 'name', '')} (@id: {getattr(field, 'id', '')})  type: {getattr(field, 'data_type', None)}")
    record_sets.append(record_set.id)
    print()

# Display first record for each record set, if available
for record_set_id in record_sets:
    print(f"--- First record from Record Set @id: {record_set_id} ---")
    try:
        for rec in dataset.records(record_set=record_set_id):
            print(json.dumps(rec, indent=2))
            break
    except Exception as ex:
        print(f"Could not load records for {record_set_id}: {ex}")
    print()

## 3. Data Extraction
Load data from each discovered record set into a pandas DataFrame for analysis. 

**Note:** The record sets and fields are referenced by their `@id` throughout.

In [ ]:
# Create DataFrames for each record set (by @id)
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from {record_set_id}")
        else:
            print(f"No records found in {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

# Show columns for the first non-empty DataFrame (if any)
for rs_id, df in dataframes.items():
    if not df.empty:
        sample_record_set_id = rs_id
        print(f"Sample DataFrame columns for record set @id: {sample_record_set_id}")
        print(df.columns.tolist())
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing, such as filtering records based on numeric criteria, normalizing a field, and grouping data. 

You should reference columns by their `@id` as demonstrated below.

In [ ]:
# EDA for the sample record set
# You may adjust numeric_field_id and group_field_id according to the actual data loaded

record_set_id = sample_record_set_id
df = dataframes[record_set_id]
print(f"Working with record set: {record_set_id}")

# Choose a numeric field by @id for EDA
numeric_cols = [col for col in df.columns if df[col].dtype.kind in 'biufc']
if numeric_cols:
    numeric_field_id = numeric_cols[0]  # pick first numeric column
    print(f"Numeric field selected by @id: {numeric_field_id}")
else:
    print("No numeric field found for EDA.")
    numeric_field_id = None

if numeric_field_id is not None:
    # Set a threshold for filtering
    threshold = df[numeric_field_id].mean()  # using mean as a sample threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)})")

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"First few normalized rows for field {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()])

    # Pick a grouping field (categorical) if available
    group_fields = [col for col in df.columns if df[col].dtype.name == 'object' and col != numeric_field_id]
    group_field_id = group_fields[0] if group_fields else None
    if group_field_id:
        print(f"Grouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. These help highlight important patterns or outliers in the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization for the selected numeric field
if numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} after filtering")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # If grouped data is available, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty and group_field_id:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This exploration demonstrated how to load a Croissant schema-based dataset with `mlcroissant`, inspect its structure (record sets and fields by `@id`), extract records into pandas DataFrames, and perform initial EDA and visualization.
- All entity references—record sets, fields, columns—were made using their `@id` fields for consistency.
- The approach is extensible to more advanced modeling, given the dataset's ordered logistic regression nature and rich socio-demographic detail.

For deeper insight, refer back to the record set and field `@id`s in the data overview and apply domain-specific analysis as needed.
